In [ ]:
# ============================================================
# MODULE 01 — INGESTION
# ============================================================
# PURPOSE:
# Safely load ONE F1 qualifying CSV and prepare it for the
# next module.
#
# IMPORTANT:
# - The original CSV is NEVER modified.
# - No cleaning or analysis happens here.
# - The dataframe is copied before anything downstream uses it.
# - Drivers are detected automatically from *_TIME columns.
#
# INPUT:
#   One qualifying CSV
#
# OUTPUT:
#   raw_df
#   work_df
#   drivers
#   ingestion_report
# ============================================================


import pandas as pd
from google.colab import files


# ------------------------------------------------------------
# STEP 01 — UPLOAD CSV
# ------------------------------------------------------------

uploaded = files.upload()

if not uploaded:
    raise ValueError("No CSV file was uploaded.")

filename = next(iter(uploaded))

print("=" * 60)
print("MODULE 01 — INGESTION")
print("=" * 60)
print(f"File: {filename}")


# ------------------------------------------------------------
# STEP 02 — READ ORIGINAL CSV
# ------------------------------------------------------------

raw_df = pd.read_csv(
    filename,
    comment="#",
    header=0
)


# ------------------------------------------------------------
# STEP 03 — CREATE WORKING COPY
# ------------------------------------------------------------
# raw_df = source copy
# work_df = copy used by later modules
#
# The original raw dataframe remains untouched.
# ------------------------------------------------------------

work_df = raw_df.copy(deep=True)


# ------------------------------------------------------------
# STEP 04 — DETECT DRIVERS
# ------------------------------------------------------------
# TracingInsights stores driver lap times in columns such as:
#
# VER_TIME
# NOR_TIME
# HAM_TIME
# RUS_TIME
#
# The driver name is everything before "_TIME".
# ------------------------------------------------------------

time_columns = [
    col for col in raw_df.columns
    if col.endswith("_TIME")
]

drivers = [
    col[:-5]
    for col in time_columns
]


if not drivers:
    raise ValueError(
        "No driver *_TIME columns were detected."
    )


# ------------------------------------------------------------
# STEP 05 — BASIC INGESTION REPORT
# ------------------------------------------------------------

ingestion_report = {
    "File": filename,
    "Rows": len(raw_df),
    "Columns": len(raw_df.columns),
    "Column_Names": list(raw_df.columns),
    "Drivers": drivers,
    "Driver_Count": len(drivers),
}


# ------------------------------------------------------------
# STEP 06 — CHECKPOINT
# ------------------------------------------------------------

print("\nINGESTION CHECKPOINT")
print("-" * 60)

print(f"Rows loaded:     {ingestion_report['Rows']}")
print(f"Columns loaded:  {ingestion_report['Columns']}")
print(f"Working copy:    CREATED")
print(f"Drivers detected: {ingestion_report['Drivers']}")

print("\nColumns:")
print(list(raw_df.columns))

print("\nModule 01 status: PASS ✅")

Saving spanish-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv to spanish-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03 (1).csv
MODULE 01 — INGESTION
File: spanish-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03 (1).csv

INGESTION CHECKPOINT
------------------------------------------------------------
Rows loaded:     19
Columns loaded:  53
Working copy:    CREATED
Drivers detected: ['VER', 'RUS', 'NOR', 'HAM']

Columns:
['LAP', 'VER_TIME', 'VER_POSITION', 'VER_POSITION_CHANGE', 'VER_COMPOUND', 'VER_STINT', 'VER_PITSTOP', 'VER_TIRE_LIFE', 'VER_S1', 'VER_S2', 'VER_S3', 'VER_SECTOR_SUM', 'VER_STATUS', 'VER_PERSONAL_BEST', 'RUS_TIME', 'RUS_POSITION', 'RUS_POSITION_CHANGE', 'RUS_COMPOUND', 'RUS_STINT', 'RUS_PITSTOP', 'RUS_TIRE_LIFE', 'RUS_S1', 'RUS_S2', 'RUS_S3', 'RUS_SECTOR_SUM', 'RUS_STATUS', 'RUS_PERSONAL_BEST', 'NOR_TIME', 'NOR_POSITION', 'NOR_POSITION_CHANGE', 'NOR_COMPOUND', 'NOR_STINT', 'NOR_PITSTOP', 'NOR

In [ ]:
# ============================================================
# BATCH CHECKPOINT — MODULE 01
# ============================================================
# PURPOSE:
# Test the already-passed Qualifying Module 01 across 8 CSV files.
#
# IMPORTANT:
# - Upload exactly 8 CSV files.
# - Original CSV files are never modified.
# - Each file gets its own raw_df and work_df copy.
# - Drivers are detected automatically from *_TIME columns.
# - This cell does NOT perform Module 02 processing.
#
# INPUT:
#   8 qualifying CSV files
#
# OUTPUT:
#   batch_results
#   batch_checkpoint
#   quali_batch_data
# ============================================================


from google.colab import files
import pandas as pd


# ------------------------------------------------------------
# STEP 01 — UPLOAD 8 QUALIFYING CSV FILES
# ------------------------------------------------------------

uploaded = files.upload()

if len(uploaded) != 8:
    raise ValueError(
        f"Expected exactly 8 qualifying CSV files, "
        f"but received {len(uploaded)}."
    )


# ------------------------------------------------------------
# STEP 02 — PROCESS EACH CSV THROUGH MODULE 01
# ------------------------------------------------------------

batch_results = []
quali_batch_data = {}


for filename in uploaded:

    try:

        # Read source CSV
        raw_df = pd.read_csv(
            filename,
            comment="#",
            header=0
        )

        # Create protected working copy
        work_df = raw_df.copy(deep=True)

        # Detect driver columns automatically
        time_columns = [
            col for col in raw_df.columns
            if col.endswith("_TIME")
        ]

        drivers = [
            col[:-5]
            for col in time_columns
        ]

        if not drivers:
            raise ValueError(
                "No driver *_TIME columns were detected."
            )

        # Store data for later modules
        quali_batch_data[filename] = {
            "raw_df": raw_df,
            "work_df": work_df,
            "drivers": drivers
        }

        # Record checkpoint
        batch_results.append({
            "Qualifying_File": filename,
            "Status": "PASS",
            "Rows": len(raw_df),
            "Columns": len(raw_df.columns),
            "Drivers": ", ".join(drivers),
            "Driver_Count": len(drivers),
            "Working_Copy": "YES"
        })

    except Exception as e:

        batch_results.append({
            "Qualifying_File": filename,
            "Status": "FAIL",
            "Rows": None,
            "Columns": None,
            "Drivers": None,
            "Driver_Count": None,
            "Working_Copy": "NO",
            "Error": str(e)
        })


# ------------------------------------------------------------
# STEP 03 — CREATE CHECKPOINT TABLE
# ------------------------------------------------------------

batch_checkpoint = pd.DataFrame(batch_results)


# ------------------------------------------------------------
# STEP 04 — DISPLAY CHECKPOINT
# ------------------------------------------------------------

print("=" * 70)
print("MODULE 01 — 8-QUALIFYING BATCH CHECKPOINT")
print("=" * 70)

display(batch_checkpoint)


# ------------------------------------------------------------
# STEP 05 — FINAL BATCH STATUS
# ------------------------------------------------------------

pass_count = (
    batch_checkpoint["Status"] == "PASS"
).sum()

fail_count = (
    batch_checkpoint["Status"] == "FAIL"
).sum()

print("\nBATCH SUMMARY")
print("-" * 70)
print(f"Files processed: {len(batch_checkpoint)}")
print(f"PASS:           {pass_count}")
print(f"FAIL:           {fail_count}")


if fail_count == 0:
    print("\nMODULE 01 — BATCH STATUS: PASS ✅")
else:
    print("\nMODULE 01 — BATCH STATUS: FAIL ❌")
    print(
        "Do not proceed to Module 02 until the "
        "failed file(s) are investigated."
    )

Saving bahrain-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv to bahrain-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv
Saving belgian-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv to belgian-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv
Saving dutch-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv to dutch-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv
Saving emilia-romagna-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv to emilia-romagna-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv
Saving hungarian-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv to hungarian-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv
Saving italian-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv to it

,Qualifying_File,Status,Rows,Columns,Drivers,Driver_Count,Working_Copy
0,bahrain-grand-prix-qualifying-4drivers-Tracing...,PASS,20,53,"LEC, RUS, SAI, VER",4,YES
1,belgian-grand-prix-qualifying-4drivers-Tracing...,PASS,23,53,"HAM, LEC, PER, VER",4,YES
2,dutch-grand-prix-qualifying-4drivers-TracingIn...,PASS,19,53,"NOR, PIA, RUS, VER",4,YES
3,emilia-romagna-grand-prix-qualifying-4drivers-...,PASS,22,53,"VER, PIA, NOR, LEC",4,YES
4,hungarian-grand-prix-qualifying-4drivers-Traci...,PASS,27,53,"NOR, PIA, VER, SAI",4,YES
5,italian-grand-prix-qualifying-4drivers-Tracing...,PASS,22,53,"LEC, NOR, PIA, RUS",4,YES
6,singapore-grand-prix-qualifying-4drivers-Traci...,PASS,19,53,"HAM, NOR, RUS, VER",4,YES
7,spanish-grand-prix-qualifying-4drivers-Tracing...,PASS,19,53,"VER, RUS, NOR, HAM",4,YES



BATCH SUMMARY
----------------------------------------------------------------------
Files processed: 8
PASS:           8
FAIL:           0

MODULE 01 — BATCH STATUS: PASS ✅


In [ ]:
# ============================================================
# MODULE 02 — NORMALIZATION
# ============================================================
# PURPOSE:
# Convert the ingested qualifying CSV into a standardized
# long-format dataframe ready for auditing and validation.
#
# IMPORTANT:
# - Uses the Module 01 working copy.
# - Original CSV is never modified.
# - No validity decisions are made here.
# - No rows are removed because of analytical validity rules.
# - Source qualifying stints are preserved.
#
# INPUT:
#   Module 01 work_df
#   Module 01 drivers
#
# OUTPUT:
#   analysis_df
#   normalization_report
# ============================================================


import pandas as pd
import numpy as np


# ------------------------------------------------------------
# STEP 01 — SELECT DEVELOPMENT QUALIFYING
# ------------------------------------------------------------
# Uses the Spanish GP already loaded by Module 01.
# ------------------------------------------------------------

test_quali = next(
    name for name in quali_batch_data
    if "spanish-grand-prix" in name.lower()
)

work_df = quali_batch_data[test_quali]["work_df"].copy(deep=True)

drivers = quali_batch_data[test_quali]["drivers"].copy()

print("=" * 60)
print("MODULE 02 — NORMALIZATION")
print("=" * 60)
print(f"Test qualifying: {test_quali}")


# ------------------------------------------------------------
# STEP 02 — NORMALIZE LAP COLUMN
# ------------------------------------------------------------

work_df["LAP"] = pd.to_numeric(
    work_df["LAP"],
    errors="coerce"
)

# Remove rows without a numeric lap number
work_df = work_df.dropna(subset=["LAP"]).copy()

# Keep session laps only
work_df = work_df[work_df["LAP"] >= 1].copy()

work_df["LAP"] = work_df["LAP"].astype(int)

work_df = work_df.sort_values("LAP").reset_index(drop=True)


# ------------------------------------------------------------
# STEP 03 — NORMALIZE DRIVER COLUMN TYPES
# ------------------------------------------------------------

numeric_suffixes = [
    "TIME",
    "POSITION",
    "POSITION_CHANGE",
    "STINT",
    "TIRE_LIFE",
    "S1",
    "S2",
    "S3",
    "SECTOR_SUM",
    "STATUS",
]

for driver in drivers:

    for suffix in numeric_suffixes:

        column = f"{driver}_{suffix}"

        if column in work_df.columns:

            work_df[column] = pd.to_numeric(
                work_df[column],
                errors="coerce"
            )

    pit_column = f"{driver}_PITSTOP"

    if pit_column in work_df.columns:

        work_df[pit_column] = (
            work_df[pit_column]
            .astype(str)
            .str.strip()
            .str.upper()
            .map({
                "TRUE": True,
                "FALSE": False
            })
        )


# ------------------------------------------------------------
# STEP 04 — BUILD LONG-FORM ANALYSIS DATA
# ------------------------------------------------------------

records = []

for driver in drivers:

    for _, row in work_df.iterrows():

        records.append({
            "Driver": driver,
            "Lap": row["LAP"],
            "LapTime": row.get(f"{driver}_TIME"),
            "Position": row.get(f"{driver}_POSITION"),
            "Position_Change": row.get(f"{driver}_POSITION_CHANGE"),
            "Compound": row.get(f"{driver}_COMPOUND"),
            "Stint": row.get(f"{driver}_STINT"),
            "PitStop": row.get(f"{driver}_PITSTOP"),
            "TireLife": row.get(f"{driver}_TIRE_LIFE"),
            "S1": row.get(f"{driver}_S1"),
            "S2": row.get(f"{driver}_S2"),
            "S3": row.get(f"{driver}_S3"),
            "SectorSum": row.get(f"{driver}_SECTOR_SUM"),
            "Status": row.get(f"{driver}_STATUS"),
            "PersonalBest": row.get(f"{driver}_PERSONAL_BEST")
        })


analysis_df = pd.DataFrame(records)


# ------------------------------------------------------------
# STEP 05 — STANDARDIZE ANALYSIS TYPES
# ------------------------------------------------------------

numeric_analysis_columns = [
    "Lap",
    "LapTime",
    "Position",
    "Position_Change",
    "Stint",
    "TireLife",
    "S1",
    "S2",
    "S3",
    "SectorSum",
    "Status",
    "PersonalBest"
]

for column in numeric_analysis_columns:

    analysis_df[column] = pd.to_numeric(
        analysis_df[column],
        errors="coerce"
    )


analysis_df["Driver"] = analysis_df["Driver"].astype(str)

analysis_df["Compound"] = (
    analysis_df["Compound"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA})
)


# ------------------------------------------------------------
# STEP 06 — NORMALIZATION CHECKPOINT
# ------------------------------------------------------------

normalization_report = {
    "Qualifying": test_quali,
    "Source_Rows": len(
        quali_batch_data[test_quali]["raw_df"]
    ),
    "Normalized_Laps": work_df["LAP"].nunique(),
    "Drivers": len(drivers),
    "Driver_List": drivers,
    "Analysis_Rows": len(analysis_df),
    "Analysis_Columns": len(analysis_df.columns),
}


print("\nNORMALIZATION CHECKPOINT")
print("-" * 60)
print(f"Drivers detected:  {normalization_report['Drivers']}")
print(f"Drivers:           {drivers}")
print(f"Laps detected:     {normalization_report['Normalized_Laps']}")
print(f"Analysis rows:     {normalization_report['Analysis_Rows']}")
print(f"Analysis columns:  {normalization_report['Analysis_Columns']}")

print("\nAnalysis dataframe preview:")
display(analysis_df.head(10))

print("\nModule 02 status: PASS ✅")

MODULE 02 — NORMALIZATION
Test qualifying: spanish-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv

NORMALIZATION CHECKPOINT
------------------------------------------------------------
Drivers detected:  4
Drivers:           ['VER', 'RUS', 'NOR', 'HAM']
Laps detected:     18
Analysis rows:     72
Analysis columns:  15

Analysis dataframe preview:


,Driver,Lap,LapTime,Position,Position_Change,Compound,Stint,PitStop,TireLife,S1,S2,S3,SectorSum,Status,PersonalBest
0,VER,1,NaN,NaN,NaN,SOFT,1.0,False,1.0,NaN,40.931,25.920,NaN,1.0,NaN
1,VER,2,72.306,NaN,NaN,SOFT,1.0,False,2.0,21.680,28.887,21.739,72.306,1.0,NaN
2,VER,3,102.809,NaN,NaN,SOFT,1.0,False,3.0,29.797,42.484,30.528,102.809,1.0,NaN
3,VER,4,NaN,NaN,NaN,SOFT,2.0,True,1.0,NaN,37.176,24.866,NaN,1.0,NaN
4,VER,5,71.653,NaN,NaN,SOFT,2.0,False,2.0,21.520,28.553,21.580,71.653,1.0,NaN
5,VER,6,95.627,NaN,NaN,SOFT,2.0,False,3.0,27.350,36.626,31.651,95.627,1.0,NaN
6,VER,7,NaN,NaN,NaN,SOFT,3.0,True,1.0,NaN,40.293,25.394,NaN,1.0,NaN
7,VER,8,71.673,NaN,NaN,SOFT,3.0,False,2.0,21.527,28.595,21.551,71.673,1.0,NaN
8,VER,9,98.172,NaN,NaN,SOFT,3.0,False,3.0,28.860,38.176,31.136,98.172,1.0,NaN
9,VER,10,NaN,NaN,NaN,SOFT,4.0,True,1.0,NaN,40.264,25.316,NaN,1.0,NaN



Module 02 status: PASS ✅


In [ ]:
# ============================================================
# MODULE 02 — NORMALIZATION
# 8-QUALIFYING BATCH CHECKPOINT
# ============================================================
# PURPOSE:
# Apply the exact Module 02 normalization logic to all
# 8 qualifying sessions.
#
# IMPORTANT:
# - Uses quali_batch_data created by Module 01.
# - NO files are uploaded again.
# - Original data remain untouched.
# - Each qualifying session gets its own normalized analysis_df.
# - Drivers come from Module 01 automatic detection.
#
# INPUT:
#   quali_batch_data
#
# OUTPUT:
#   quali_normalized_batch_data
#   normalization_batch_checkpoint
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# STEP 1 — REUSABLE NORMALIZATION FUNCTION
# ------------------------------------------------------------

def normalize_quali(work_df, drivers, quali_name):

    # Always work on a fresh copy
    work_df = work_df.copy(deep=True)


    # ----- LAP NORMALIZATION -----

    work_df["LAP"] = pd.to_numeric(
        work_df["LAP"],
        errors="coerce"
    )

    work_df = work_df.dropna(
        subset=["LAP"]
    ).copy()

    work_df = work_df[
        work_df["LAP"] >= 1
    ].copy()

    work_df["LAP"] = work_df["LAP"].astype(int)

    work_df = (
        work_df
        .sort_values("LAP")
        .reset_index(drop=True)
    )


    # ----- NUMERIC CONVERSION -----

    numeric_suffixes = [
        "TIME",
        "POSITION",
        "POSITION_CHANGE",
        "STINT",
        "TIRE_LIFE",
        "S1",
        "S2",
        "S3",
        "SECTOR_SUM",
        "STATUS",
    ]

    for driver in drivers:

        for suffix in numeric_suffixes:

            column = f"{driver}_{suffix}"

            if column in work_df.columns:

                work_df[column] = pd.to_numeric(
                    work_df[column],
                    errors="coerce"
                )


        # ----- PIT STOP CONVERSION -----

        pit_column = f"{driver}_PITSTOP"

        if pit_column in work_df.columns:

            work_df[pit_column] = (
                work_df[pit_column]
                .astype(str)
                .str.strip()
                .str.upper()
                .map({
                    "TRUE": True,
                    "FALSE": False
                })
            )


    # --------------------------------------------------------
    # STEP 2 — WIDE → LONG FORMAT
    # --------------------------------------------------------

    records = []

    for driver in drivers:

        for _, row in work_df.iterrows():

            records.append({
                "Driver": driver,
                "Lap": row["LAP"],
                "LapTime": row.get(
                    f"{driver}_TIME"
                ),
                "Position": row.get(
                    f"{driver}_POSITION"
                ),
                "Position_Change": row.get(
                    f"{driver}_POSITION_CHANGE"
                ),
                "Compound": row.get(
                    f"{driver}_COMPOUND"
                ),
                "Stint": row.get(
                    f"{driver}_STINT"
                ),
                "PitStop": row.get(
                    f"{driver}_PITSTOP"
                ),
                "TireLife": row.get(
                    f"{driver}_TIRE_LIFE"
                ),
                "S1": row.get(
                    f"{driver}_S1"
                ),
                "S2": row.get(
                    f"{driver}_S2"
                ),
                "S3": row.get(
                    f"{driver}_S3"
                ),
                "SectorSum": row.get(
                    f"{driver}_SECTOR_SUM"
                ),
                "Status": row.get(
                    f"{driver}_STATUS"
                ),
                "PersonalBest": row.get(
                    f"{driver}_PERSONAL_BEST"
                )
            })


    analysis_df = pd.DataFrame(records)


    # --------------------------------------------------------
    # STEP 3 — FINAL COLUMN NORMALIZATION
    # --------------------------------------------------------

    numeric_analysis_columns = [
        "Lap",
        "LapTime",
        "Position",
        "Position_Change",
        "Stint",
        "TireLife",
        "S1",
        "S2",
        "S3",
        "SectorSum",
        "Status",
        "PersonalBest"
    ]

    for column in numeric_analysis_columns:

        analysis_df[column] = pd.to_numeric(
            analysis_df[column],
            errors="coerce"
        )


    analysis_df["Driver"] = (
        analysis_df["Driver"]
        .astype(str)
    )

    analysis_df["Compound"] = (
        analysis_df["Compound"]
        .astype("string")
        .str.strip()
        .replace({"": pd.NA})
    )


    # --------------------------------------------------------
    # STEP 4 — QUALIFYING REPORT
    # --------------------------------------------------------

    report = {
        "Qualifying_File": quali_name,
        "Source_Rows": len(work_df),
        "Normalized_Laps": work_df["LAP"].nunique(),
        "Drivers": len(drivers),
        "Driver_List": drivers,
        "Analysis_Rows": len(analysis_df),
        "Analysis_Columns": len(analysis_df.columns),
    }

    return analysis_df, report


# ============================================================
# STEP 5 — RUN ALL 8 QUALIFYING SESSIONS
# ============================================================

quali_normalized_batch_data = {}
batch_results = []


for quali_name, quali_data in quali_batch_data.items():

    try:

        analysis_df, report = normalize_quali(
            quali_data["work_df"],
            quali_data["drivers"],
            quali_name
        )

        quali_normalized_batch_data[quali_name] = {
            "analysis_df": analysis_df,
            "report": report
        }

        batch_results.append({
            "Qualifying_File": quali_name,
            "Status": "PASS",
            "Laps": report["Normalized_Laps"],
            "Drivers": report["Drivers"],
            "Analysis_Rows": report["Analysis_Rows"],
            "Analysis_Columns": report["Analysis_Columns"]
        })


    except Exception as e:

        batch_results.append({
            "Qualifying_File": quali_name,
            "Status": "FAIL",
            "Laps": None,
            "Drivers": None,
            "Analysis_Rows": None,
            "Analysis_Columns": None,
            "Error": str(e)
        })


# ============================================================
# STEP 6 — BATCH CHECKPOINT
# ============================================================

normalization_batch_checkpoint = pd.DataFrame(
    batch_results
)


print("=" * 75)
print("MODULE 02 — 8-QUALIFYING BATCH CHECKPOINT")
print("=" * 75)

display(normalization_batch_checkpoint)


pass_count = (
    normalization_batch_checkpoint["Status"] == "PASS"
).sum()

fail_count = (
    normalization_batch_checkpoint["Status"] == "FAIL"
).sum()


print("\nBATCH SUMMARY")
print("-" * 75)
print(
    f"Files processed: "
    f"{len(normalization_batch_checkpoint)}"
)

print(f"PASS:            {pass_count}")
print(f"FAIL:            {fail_count}")


if fail_count == 0:

    print("\nMODULE 02 — BATCH STATUS: PASS ✅")

else:

    print("\nMODULE 02 — BATCH STATUS: FAIL ❌")

    print(
        "Do not proceed to Module 03 until "
        "the failed qualifying session(s) are investigated."
    )

MODULE 02 — 8-QUALIFYING BATCH CHECKPOINT


,Qualifying_File,Status,Laps,Drivers,Analysis_Rows,Analysis_Columns
0,bahrain-grand-prix-qualifying-4drivers-Tracing...,PASS,19,4,76,15
1,belgian-grand-prix-qualifying-4drivers-Tracing...,PASS,22,4,88,15
2,dutch-grand-prix-qualifying-4drivers-TracingIn...,PASS,18,4,72,15
3,emilia-romagna-grand-prix-qualifying-4drivers-...,PASS,21,4,84,15
4,hungarian-grand-prix-qualifying-4drivers-Traci...,PASS,26,4,104,15
5,italian-grand-prix-qualifying-4drivers-Tracing...,PASS,21,4,84,15
6,singapore-grand-prix-qualifying-4drivers-Traci...,PASS,18,4,72,15
7,spanish-grand-prix-qualifying-4drivers-Tracing...,PASS,18,4,72,15



BATCH SUMMARY
---------------------------------------------------------------------------
Files processed: 8
PASS:            8
FAIL:            0

MODULE 02 — BATCH STATUS: PASS ✅


In [ ]:
# ============================================================
# MODULE 03 — AUDIT
# SPANISH GP QUALIFYING REFERENCE TEST
# ============================================================
# PURPOSE:
# Check whether the normalized qualifying data are internally
# consistent and describe the structure of the session.
#
# IMPORTANT:
# - Audit does NOT decide analytical validity.
# - Audit does NOT remove rows.
# - Missing LapTimes are recorded, not rejected.
# - Transition-only stints are recorded, not rejected.
# - Original data remain untouched.
#
# INPUT:
#   quali_normalized_batch_data
#
# OUTPUT:
#   audited_quali_analysis_df
#   quali_boundary_audit
#   quali_stint_audit
#   quali_audit_report
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# STEP 1 — SELECT REFERENCE QUALIFYING SESSION
# ------------------------------------------------------------

test_quali = next(
    name for name in quali_normalized_batch_data
    if "spanish-grand-prix" in name.lower()
)

analysis_df = (
    quali_normalized_batch_data[test_quali]["analysis_df"]
    .copy(deep=True)
)

print("=" * 70)
print("MODULE 03 — QUALIFYING AUDIT")
print("=" * 70)
print(f"Test qualifying: {test_quali}")


# ------------------------------------------------------------
# STEP 2 — SECTOR INTEGRITY AUDIT
# ------------------------------------------------------------

sector_available = (
    analysis_df["S1"].notna()
    & analysis_df["S2"].notna()
    & analysis_df["S3"].notna()
    & analysis_df["SectorSum"].notna()
)

analysis_df["Sector_Check"] = pd.NA

analysis_df.loc[
    sector_available,
    "Sector_Check"
] = np.isclose(
    (
        analysis_df.loc[sector_available, "S1"]
        + analysis_df.loc[sector_available, "S2"]
        + analysis_df.loc[sector_available, "S3"]
    ),
    analysis_df.loc[sector_available, "SectorSum"],
    atol=0.001
)

sector_checked = sector_available.sum()

sector_pass = (
    analysis_df["Sector_Check"] == True
).sum()

sector_fail = (
    analysis_df["Sector_Check"] == False
).sum()


# ------------------------------------------------------------
# STEP 3 — STRUCTURAL / BOUNDARY AUDIT
# ------------------------------------------------------------

audit_df = (
    analysis_df
    .sort_values(["Driver", "Lap"])
    .copy()
)

audit_df["Previous_Lap"] = (
    audit_df
    .groupby("Driver")["Lap"]
    .shift(1)
)

audit_df["Previous_Stint"] = (
    audit_df
    .groupby("Driver")["Stint"]
    .shift(1)
)

audit_df["Lap_Gap"] = (
    audit_df["Lap"] - audit_df["Previous_Lap"]
)

audit_df["Stint_Decrease"] = (
    audit_df["Stint"] < audit_df["Previous_Stint"]
)

audit_df["Stint_Transition"] = (
    audit_df["Stint"] != audit_df["Previous_Stint"]
)

audit_df["PitStop_Without_Transition"] = (
    (audit_df["PitStop"] == True)
    & (audit_df["Stint_Transition"] == False)
)


# ------------------------------------------------------------
# STEP 4 — AUDIT FLAGS
# ------------------------------------------------------------

audit_df["Missing_LapTime"] = (
    audit_df["LapTime"].isna()
)

audit_df["Missing_Stint"] = (
    audit_df["Stint"].isna()
)

audit_df["Missing_Compound"] = (
    audit_df["Compound"].isna()
)

audit_df["Lap_Sequence_Gap"] = (
    audit_df["Previous_Lap"].notna()
    & (audit_df["Lap_Gap"] > 1)
)


# ------------------------------------------------------------
# STEP 5 — TRANSITION-ONLY STINT IDENTIFICATION
# ------------------------------------------------------------
# A transition-only stint is a source stint containing
# no LapTime values.
#
# This is descriptive only.
# It is NOT a rejection rule.
# ------------------------------------------------------------

audit_df["LapTime_Available"] = (
    audit_df["LapTime"].notna()
)

stint_laptime_counts = (
    audit_df
    .groupby(["Driver", "Stint"], dropna=False)["LapTime"]
    .count()
    .reset_index(name="LapTimes_Available")
)

stint_laptime_counts["Transition_Only_Stint"] = (
    stint_laptime_counts["LapTimes_Available"] == 0
)


# ------------------------------------------------------------
# STEP 6 — BOUNDARY AUDIT SUMMARY
# ------------------------------------------------------------

audit_df["Boundary_Audit_OK"] = ~(
    audit_df["Missing_LapTime"]
    | audit_df["Missing_Stint"]
    | audit_df["Missing_Compound"]
    | audit_df["Lap_Sequence_Gap"]
    | audit_df["Stint_Decrease"]
    | audit_df["PitStop_Without_Transition"]
)


# ------------------------------------------------------------
# STEP 7 — BOUNDARY AUDIT TABLE
# ------------------------------------------------------------

quali_boundary_audit = pd.DataFrame({
    "Check": [
        "Missing LapTime",
        "Missing Stint",
        "Missing Compound",
        "Lap Sequence Gap",
        "Stint Number Decrease",
        "PitStop Without Stint Transition"
    ],

    "Failures": [
        audit_df["Missing_LapTime"].sum(),
        audit_df["Missing_Stint"].sum(),
        audit_df["Missing_Compound"].sum(),
        audit_df["Lap_Sequence_Gap"].sum(),
        audit_df["Stint_Decrease"].sum(),
        audit_df["PitStop_Without_Transition"].sum()
    ]
})


# ------------------------------------------------------------
# STEP 8 — STINT AUDIT
# ------------------------------------------------------------

quali_stint_audit = (
    audit_df
    .groupby(["Driver", "Stint"], dropna=False)
    .agg(
        Start_Lap=("Lap", "min"),
        End_Lap=("Lap", "max"),
        Laps=("Lap", "count"),
        LapTimes_Available=("LapTime", "count"),
        Missing_LapTime_Count=("LapTime", lambda x: x.isna().sum()),
        Compounds=("Compound", "nunique"),
        Fastest_LapTime=("LapTime", "min")
    )
    .reset_index()
)

quali_stint_audit["Missing_LapTime"] = (
    quali_stint_audit["Missing_LapTime_Count"] > 0
)

quali_stint_audit["Transition_Only_Stint"] = (
    quali_stint_audit["LapTimes_Available"] == 0
)


# ------------------------------------------------------------
# STEP 9 — FINAL AUDIT REPORT
# ------------------------------------------------------------

quali_audit_report = {
    "Qualifying_File": test_quali,
    "Rows_Audited": len(audit_df),
    "Drivers": audit_df["Driver"].nunique(),

    "Session_Laps": audit_df["Lap"].nunique(),

    "Sector_Rows_Checked": int(sector_checked),
    "Sector_Pass": int(sector_pass),
    "Sector_Fail": int(sector_fail),

    "Boundary_Failures": int(
        (~audit_df["Boundary_Audit_OK"]).sum()
    ),

    "Stints_Audited": len(quali_stint_audit),

    "Transition_Only_Stints": int(
        quali_stint_audit["Transition_Only_Stint"].sum()
    )
}


# ------------------------------------------------------------
# STEP 10 — OUTPUT
# ------------------------------------------------------------

audited_quali_analysis_df = (
    audit_df.copy(deep=True)
)

print("\nSECTOR INTEGRITY")
print("-" * 70)
print(f"Rows checked: {sector_checked}")
print(f"PASS:         {sector_pass}")
print(f"FAIL:         {sector_fail}")

print("\nBOUNDARY / STRUCTURAL AUDIT")
print("-" * 70)
display(quali_boundary_audit)

print("\nSTINT AUDIT")
print("-" * 70)
display(quali_stint_audit)

print("\nAUDIT SUMMARY")
print("-" * 70)
print(f"Rows audited:          {quali_audit_report['Rows_Audited']}")
print(f"Drivers:               {quali_audit_report['Drivers']}")
print(f"Session laps:          {quali_audit_report['Session_Laps']}")
print(f"Sector failures:       {quali_audit_report['Sector_Fail']}")
print(f"Boundary failures:     {quali_audit_report['Boundary_Failures']}")
print(f"Stints audited:        {quali_audit_report['Stints_Audited']}")
print(
    f"Transition-only stints: "
    f"{quali_audit_report['Transition_Only_Stints']}"
)

print("\nModule 03 qualifying reference test complete.")

MODULE 03 — QUALIFYING AUDIT
Test qualifying: spanish-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv

SECTOR INTEGRITY
----------------------------------------------------------------------
Rows checked: 36
PASS:         36
FAIL:         0

BOUNDARY / STRUCTURAL AUDIT
----------------------------------------------------------------------


,Check,Failures
0,Missing LapTime,37
1,Missing Stint,14
2,Missing Compound,14
3,Lap Sequence Gap,0
4,Stint Number Decrease,0
5,PitStop Without Stint Transition,0



STINT AUDIT
----------------------------------------------------------------------


,Driver,Stint,Start_Lap,End_Lap,Laps,LapTimes_Available,Missing_LapTime_Count,Compounds,Fastest_LapTime,Missing_LapTime,Transition_Only_Stint
0,HAM,1.0,1,3,3,2,1,1,73.033,True,False
1,HAM,2.0,4,6,3,2,1,1,72.143,True,False
2,HAM,3.0,7,9,3,2,1,1,72.666,True,False
3,HAM,4.0,10,12,3,2,1,1,71.792,True,False
4,HAM,5.0,13,15,3,2,1,1,71.816,True,False
5,HAM,6.0,16,18,3,1,2,1,71.701,True,False
6,NOR,1.0,1,3,3,2,1,1,72.386,True,False
7,NOR,2.0,4,6,3,2,1,1,71.872,True,False
8,NOR,3.0,7,9,3,2,1,1,71.796,True,False
9,NOR,4.0,10,12,3,2,1,1,71.383,True,False



AUDIT SUMMARY
----------------------------------------------------------------------
Rows audited:          72
Drivers:               4
Session laps:          18
Sector failures:       0
Boundary failures:     37
Stints audited:        23
Transition-only stints: 4

Module 03 qualifying reference test complete.


In [ ]:
# ============================================================
# MODULE 03 — AUDIT
# 8-QUALIFYING BATCH CHECKPOINT
# ============================================================
# PURPOSE:
# Apply the passed Module 03 qualifying audit to all
# 8 qualifying sessions.
#
# IMPORTANT:
# - Uses quali_normalized_batch_data from Module 02.
# - NO files are uploaded again.
# - Original data remain untouched.
# - Audit does NOT remove rows.
# - Missing LapTimes are recorded, not rejected.
# - Transition-only stints are recorded, not rejected.
#
# INPUT:
#   quali_normalized_batch_data
#
# OUTPUT:
#   quali_audited_batch_data
#   audit_batch_checkpoint
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# STEP 1 — AUDIT FUNCTION
# ------------------------------------------------------------

def audit_quali(analysis_df, quali_name):

    analysis_df = analysis_df.copy(deep=True)


    # --------------------------------------------------------
    # SECTOR INTEGRITY
    # --------------------------------------------------------

    sector_available = (
        analysis_df["S1"].notna()
        & analysis_df["S2"].notna()
        & analysis_df["S3"].notna()
        & analysis_df["SectorSum"].notna()
    )

    analysis_df["Sector_Check"] = pd.NA

    analysis_df.loc[
        sector_available,
        "Sector_Check"
    ] = np.isclose(
        (
            analysis_df.loc[sector_available, "S1"]
            + analysis_df.loc[sector_available, "S2"]
            + analysis_df.loc[sector_available, "S3"]
        ),
        analysis_df.loc[sector_available, "SectorSum"],
        atol=0.001
    )

    sector_checked = sector_available.sum()

    sector_pass = (
        analysis_df["Sector_Check"] == True
    ).sum()

    sector_fail = (
        analysis_df["Sector_Check"] == False
    ).sum()


    # --------------------------------------------------------
    # STRUCTURAL / BOUNDARY AUDIT
    # --------------------------------------------------------

    audit_df = (
        analysis_df
        .sort_values(["Driver", "Lap"])
        .copy()
    )

    audit_df["Previous_Lap"] = (
        audit_df
        .groupby("Driver")["Lap"]
        .shift(1)
    )

    audit_df["Previous_Stint"] = (
        audit_df
        .groupby("Driver")["Stint"]
        .shift(1)
    )

    audit_df["Lap_Gap"] = (
        audit_df["Lap"]
        - audit_df["Previous_Lap"]
    )

    audit_df["Stint_Decrease"] = (
        audit_df["Stint"]
        < audit_df["Previous_Stint"]
    )

    audit_df["Stint_Transition"] = (
        audit_df["Stint"]
        != audit_df["Previous_Stint"]
    )

    audit_df["PitStop_Without_Transition"] = (
        (audit_df["PitStop"] == True)
        & (audit_df["Stint_Transition"] == False)
    )


    # --------------------------------------------------------
    # AUDIT FLAGS
    # --------------------------------------------------------

    audit_df["Missing_LapTime"] = (
        audit_df["LapTime"].isna()
    )

    audit_df["Missing_Stint"] = (
        audit_df["Stint"].isna()
    )

    audit_df["Missing_Compound"] = (
        audit_df["Compound"].isna()
    )

    audit_df["Lap_Sequence_Gap"] = (
        audit_df["Previous_Lap"].notna()
        & (audit_df["Lap_Gap"] > 1)
    )


    # --------------------------------------------------------
    # TRANSITION-ONLY STINTS
    # --------------------------------------------------------

    stint_laptime_counts = (
        audit_df
        .groupby(
            ["Driver", "Stint"],
            dropna=False
        )["LapTime"]
        .count()
        .reset_index(
            name="LapTimes_Available"
        )
    )

    stint_laptime_counts[
        "Transition_Only_Stint"
    ] = (
        stint_laptime_counts["LapTimes_Available"] == 0
    )


    # --------------------------------------------------------
    # BOUNDARY AUDIT SUMMARY
    # --------------------------------------------------------

    audit_df["Boundary_Audit_OK"] = ~(
        audit_df["Missing_LapTime"]
        | audit_df["Missing_Stint"]
        | audit_df["Missing_Compound"]
        | audit_df["Lap_Sequence_Gap"]
        | audit_df["Stint_Decrease"]
        | audit_df["PitStop_Without_Transition"]
    )


    # --------------------------------------------------------
    # STINT AUDIT
    # --------------------------------------------------------

    stint_audit = (
        audit_df
        .groupby(
            ["Driver", "Stint"],
            dropna=False
        )
        .agg(
            Start_Lap=("Lap", "min"),
            End_Lap=("Lap", "max"),
            Laps=("Lap", "count"),
            LapTimes_Available=("LapTime", "count"),
            Missing_LapTime_Count=(
                "LapTime",
                lambda x: x.isna().sum()
            ),
            Compounds=("Compound", "nunique"),
            Fastest_LapTime=("LapTime", "min")
        )
        .reset_index()
    )

    stint_audit["Missing_LapTime"] = (
        stint_audit["Missing_LapTime_Count"] > 0
    )

    stint_audit["Transition_Only_Stint"] = (
        stint_audit["LapTimes_Available"] == 0
    )


    # --------------------------------------------------------
    # REPORT
    # --------------------------------------------------------

    audit_report = {
        "Qualifying_File": quali_name,
        "Rows_Audited": len(audit_df),
        "Drivers": audit_df["Driver"].nunique(),
        "Session_Laps": audit_df["Lap"].nunique(),

        "Sector_Rows_Checked": int(
            sector_checked
        ),

        "Sector_Pass": int(
            sector_pass
        ),

        "Sector_Fail": int(
            sector_fail
        ),

        "Boundary_Failures": int(
            (~audit_df["Boundary_Audit_OK"]).sum()
        ),

        "Stints_Audited": len(
            stint_audit
        ),

        "Transition_Only_Stints": int(
            stint_audit[
                "Transition_Only_Stint"
            ].sum()
        )
    }


    return (
        audit_df,
        stint_audit,
        audit_report
    )


# ------------------------------------------------------------
# STEP 2 — RUN ALL 8 QUALIFYING SESSIONS
# ------------------------------------------------------------

quali_audited_batch_data = {}
batch_results = []


for quali_name, quali_data in (
    quali_normalized_batch_data.items()
):

    try:

        audit_df, stint_audit, report = audit_quali(
            quali_data["analysis_df"],
            quali_name
        )

        quali_audited_batch_data[quali_name] = {
            "analysis_df": audit_df,
            "stint_audit": stint_audit,
            "report": report
        }

        batch_results.append({
            "Qualifying_File": quali_name,
            "Status": "PASS",
            "Rows_Audited": report["Rows_Audited"],
            "Drivers": report["Drivers"],
            "Session_Laps": report["Session_Laps"],
            "Sector_Checked": report["Sector_Rows_Checked"],
            "Sector_Fail": report["Sector_Fail"],
            "Boundary_Failures": report["Boundary_Failures"],
            "Stints_Audited": report["Stints_Audited"],
            "Transition_Only_Stints": (
                report["Transition_Only_Stints"]
            )
        })

    except Exception as e:

        batch_results.append({
            "Qualifying_File": quali_name,
            "Status": "FAIL",
            "Rows_Audited": None,
            "Drivers": None,
            "Session_Laps": None,
            "Sector_Checked": None,
            "Sector_Fail": None,
            "Boundary_Failures": None,
            "Stints_Audited": None,
            "Transition_Only_Stints": None,
            "Error": str(e)
        })


# ------------------------------------------------------------
# STEP 3 — BATCH CHECKPOINT
# ------------------------------------------------------------

audit_batch_checkpoint = pd.DataFrame(
    batch_results
)

print("=" * 75)
print("MODULE 03 — 8-QUALIFYING BATCH CHECKPOINT")
print("=" * 75)

display(audit_batch_checkpoint)


pass_count = (
    audit_batch_checkpoint["Status"] == "PASS"
).sum()

fail_count = (
    audit_batch_checkpoint["Status"] == "FAIL"
).sum()


print("\nBATCH SUMMARY")
print("-" * 75)
print(
    f"Files processed: "
    f"{len(audit_batch_checkpoint)}"
)

print(f"PASS:            {pass_count}")
print(f"FAIL:            {fail_count}")


if fail_count == 0:

    print(
        "\nMODULE 03 — BATCH STATUS: PASS ✅"
    )

else:

    print(
        "\nMODULE 03 — BATCH STATUS: FAIL ❌"
    )

    print(
        "Do not proceed to Module 04 until "
        "the failed qualifying session(s) are investigated."
    )

MODULE 03 — 8-QUALIFYING BATCH CHECKPOINT


,Qualifying_File,Status,Rows_Audited,Drivers,Session_Laps,Sector_Checked,Sector_Fail,Boundary_Failures,Stints_Audited,Transition_Only_Stints
0,bahrain-grand-prix-qualifying-4drivers-Tracing...,PASS,76,4,19,45,0,31,28,4
1,belgian-grand-prix-qualifying-4drivers-Tracing...,PASS,88,4,22,70,0,33,23,2
2,dutch-grand-prix-qualifying-4drivers-TracingIn...,PASS,72,4,18,39,0,33,24,3
3,emilia-romagna-grand-prix-qualifying-4drivers-...,PASS,84,4,21,45,0,39,28,5
4,hungarian-grand-prix-qualifying-4drivers-Traci...,PASS,104,4,26,55,0,49,31,6
5,italian-grand-prix-qualifying-4drivers-Tracing...,PASS,84,4,21,44,0,40,26,3
6,singapore-grand-prix-qualifying-4drivers-Traci...,PASS,72,4,18,34,0,41,27,6
7,spanish-grand-prix-qualifying-4drivers-Tracing...,PASS,72,4,18,36,0,37,23,4



BATCH SUMMARY
---------------------------------------------------------------------------
Files processed: 8
PASS:            8
FAIL:            0

MODULE 03 — BATCH STATUS: PASS ✅


In [ ]:
# ============================================================
# MODULE 04 — QUALIFYING VALIDATION
# ============================================================

def validate_quali(audit_df):

    df = audit_df.copy(deep=True)

    # --------------------------------------------------------
    # 1. INITIAL STATUS
    # --------------------------------------------------------

    df["Analytical_Eligible"] = True
    df["Validation_Reason"] = "Valid"

    # --------------------------------------------------------
    # 2. LAPTIME AVAILABILITY
    # --------------------------------------------------------
    # LapTime is the primary performance observation.
    # If it does not exist, the row cannot contribute to
    # any performance metric.

    missing_laptime = df["LapTime"].isna()

    df.loc[missing_laptime, "Analytical_Eligible"] = False
    df.loc[
        missing_laptime,
        "Validation_Reason"
    ] = "Missing_LapTime"

    # --------------------------------------------------------
    # 3. TRACK STATUS
    # --------------------------------------------------------
    # Only clear-track LapTimes are eligible.
    # Missing LapTimes are already excluded above.

    non_clear_status = (
        df["LapTime"].notna() &
        (df["Status"] != 1)
    )

    df.loc[non_clear_status, "Analytical_Eligible"] = False
    df.loc[
        non_clear_status,
        "Validation_Reason"
    ] = "Non_Clear_Track_Status"

    # --------------------------------------------------------
    # 4. PIT-STOP LAP
    # --------------------------------------------------------
    # A LapTime explicitly marked as a pit-stop lap is not
    # treated as a normal performance observation.

    pitstop_lap = (
        df["LapTime"].notna() &
        (df["PitStop"] == True)
    )

    df.loc[pitstop_lap, "Analytical_Eligible"] = False
    df.loc[
        pitstop_lap,
        "Validation_Reason"
    ] = "PitStop_Lap"

    # --------------------------------------------------------
    # 5. SOURCE STINT INFORMATION
    # --------------------------------------------------------
    # STINT does NOT determine whether a LapTime is valid.
    # It only determines whether that LapTime can be assigned
    # to a Driver × Stint performance opportunity.

    df["Stint_Assigned"] = df["Stint"].notna()

    # Eligible LapTime with no source stint.
    df["Lap_Outside_Source_Stint"] = (
        df["Analytical_Eligible"] &
        ~df["Stint_Assigned"]
    )

    # Eligible LapTime WITH a source stint.
    df["Analytical_Eligible_Stint"] = (
        df["Analytical_Eligible"] &
        df["Stint_Assigned"]
    )

    # --------------------------------------------------------
    # 6. SECTOR CHECK
    # --------------------------------------------------------
    # Sector mismatch remains diagnostic.
    # It does NOT automatically invalidate the LapTime.

    df["Sector_Anomaly"] = (
        df["Sector_Check"] == False
    )

    # --------------------------------------------------------
    # 7. TRANSITION-ONLY INFORMATION
    # --------------------------------------------------------
    # A transition-only source stint contains no LapTimes.
    # Those rows are therefore already excluded by the
    # Missing_LapTime rule.
    #
    # We preserve the information in the M03 stint audit.
    # We do not need to merge it into every row here.

    # --------------------------------------------------------
    # 8. VALIDATION SUMMARY
    # --------------------------------------------------------

    report = {
        "Rows_Audited": len(df),

        "Drivers": df["Driver"].nunique(),

        "Session_Laps": df["Lap"].nunique(),

        "Analytically_Eligible_Laps": int(
            df["Analytical_Eligible"].sum()
        ),

        "Stint_Eligible_Laps": int(
            df["Analytical_Eligible_Stint"].sum()
        ),

        "Laps_Outside_Source_Stint": int(
            df["Lap_Outside_Source_Stint"].sum()
        ),

        "Missing_LapTime": int(
            missing_laptime.sum()
        ),

        "Non_Clear_Status": int(
            non_clear_status.sum()
        ),

        "PitStop_Laps": int(
            pitstop_lap.sum()
        ),

        "Sector_Anomalies": int(
            df["Sector_Anomaly"].sum()
        )
    }

    return df, report


# ============================================================
# RUN MODULE 04 — SPANISH REFERENCE
# ============================================================

spanish_quali_audit = quali_audited_batch_data[
    [name for name in quali_audited_batch_data
     if "spanish" in name.lower()][0]
]

validated_quali_df, validation_report = validate_quali(
    spanish_quali_audit["analysis_df"]
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 67)
print("MODULE 04 — QUALIFYING VALIDATION")
print("=" * 67)

print("\nVALIDATION SUMMARY")
print("-" * 67)

for key, value in validation_report.items():
    print(f"{key}: {value}")

print("\nVALIDATION REASON COUNTS")
print("-" * 67)

print(
    validated_quali_df[
        "Validation_Reason"
    ].value_counts(dropna=False)
)

print("\nLAPS OUTSIDE SOURCE STINT")
print("-" * 67)

display(
    validated_quali_df[
        validated_quali_df["Lap_Outside_Source_Stint"]
    ][
        [
            "Driver",
            "Lap",
            "LapTime",
            "Stint",
            "Status",
            "PitStop",
            "Sector_Anomaly",
            "Analytical_Eligible",
            "Analytical_Eligible_Stint",
            "Validation_Reason"
        ]
    ]
)

print("\nANALYTICAL ELIGIBILITY")
print("-" * 67)

display(
    validated_quali_df[
        [
            "Driver",
            "Lap",
            "LapTime",
            "Stint",
            "Status",
            "PitStop",
            "Sector_Anomaly",
            "Stint_Assigned",
            "Lap_Outside_Source_Stint",
            "Analytical_Eligible",
            "Analytical_Eligible_Stint",
            "Validation_Reason"
        ]
    ]
)

MODULE 04 — QUALIFYING VALIDATION

VALIDATION SUMMARY
-------------------------------------------------------------------
Rows_Audited: 72
Drivers: 4
Session_Laps: 18
Analytically_Eligible_Laps: 34
Stint_Eligible_Laps: 34
Laps_Outside_Source_Stint: 0
Missing_LapTime: 37
Non_Clear_Status: 1
PitStop_Laps: 0
Sector_Anomalies: 0

VALIDATION REASON COUNTS
-------------------------------------------------------------------
Validation_Reason
Missing_LapTime           37
Valid                     34
Non_Clear_Track_Status     1
Name: count, dtype: int64

LAPS OUTSIDE SOURCE STINT
-------------------------------------------------------------------


,Driver,Lap,LapTime,Stint,Status,PitStop,Sector_Anomaly,Analytical_Eligible,Analytical_Eligible_Stint,Validation_Reason



ANALYTICAL ELIGIBILITY
-------------------------------------------------------------------


,Driver,Lap,LapTime,Stint,Status,PitStop,Sector_Anomaly,Stint_Assigned,Lap_Outside_Source_Stint,Analytical_Eligible,Analytical_Eligible_Stint,Validation_Reason
54,HAM,1,NaN,1.0,1.0,False,False,True,False,False,False,Missing_LapTime
55,HAM,2,73.033,1.0,1.0,False,False,True,False,True,True,Valid
56,HAM,3,105.965,1.0,1.0,False,False,True,False,True,True,Valid
57,HAM,4,NaN,2.0,1.0,True,False,True,False,False,False,Missing_LapTime
58,HAM,5,72.143,2.0,1.0,False,False,True,False,True,True,Valid
...,...,...,...,...,...,...,...,...,...,...,...,...
13,VER,14,NaN,NaN,NaN,NaN,False,False,False,False,False,Missing_LapTime
14,VER,15,NaN,NaN,NaN,NaN,False,False,False,False,False,Missing_LapTime
15,VER,16,NaN,NaN,NaN,NaN,False,False,False,False,False,Missing_LapTime
16,VER,17,NaN,NaN,NaN,NaN,False,False,False,False,False,Missing_LapTime


In [ ]:
# ============================================================
# MODULE 04 — 8-QUALIFYING BATCH CHECKPOINT
# ============================================================

quali_validated_batch_data = {}
validation_batch_results = []


# ============================================================
# BATCH VALIDATION
# ============================================================

for quali_name, quali_data in quali_audited_batch_data.items():

    audit_df = quali_data["analysis_df"]

    validated_df, report = validate_quali(audit_df)

    quali_validated_batch_data[quali_name] = {
        "analysis_df": validated_df,
        "stint_audit": quali_data["stint_audit"],
        "report": report
    }

    validation_batch_results.append({
        "Qualifying_File": quali_name,
        "Status": "PASS",
        "Rows_Audited": report["Rows_Audited"],
        "Drivers": report["Drivers"],
        "Session_Laps": report["Session_Laps"],
        "Analytically_Eligible_Laps": report[
            "Analytically_Eligible_Laps"
        ],
        "Stint_Eligible_Laps": report[
            "Stint_Eligible_Laps"
        ],
        "Laps_Outside_Source_Stint": report[
            "Laps_Outside_Source_Stint"
        ],
        "Missing_LapTime": report[
            "Missing_LapTime"
        ],
        "Non_Clear_Status": report[
            "Non_Clear_Status"
        ],
        "PitStop_Laps": report[
            "PitStop_Laps"
        ],
        "Sector_Anomalies": report[
            "Sector_Anomalies"
        ]
    })


# ============================================================
# BATCH CHECKPOINT TABLE
# ============================================================

validation_batch_checkpoint = pd.DataFrame(
    validation_batch_results
)


# ============================================================
# BATCH STATUS
# ============================================================

pass_count = (
    validation_batch_checkpoint["Status"] == "PASS"
).sum()

fail_count = (
    validation_batch_checkpoint["Status"] == "FAIL"
).sum()


print("=" * 67)
print("MODULE 04 — 8-QUALIFYING BATCH CHECKPOINT")
print("=" * 67)

display(validation_batch_checkpoint)

print("\nBATCH SUMMARY")
print("-" * 67)
print(f"Files processed: {len(validation_batch_checkpoint)}")
print(f"PASS:            {pass_count}")
print(f"FAIL:            {fail_count}")

if fail_count == 0 and pass_count == 8:
    print("\nMODULE 04 — BATCH STATUS: PASS ✅")
else:
    print("\nMODULE 04 — BATCH STATUS: REVIEW ⚠️")

MODULE 04 — 8-QUALIFYING BATCH CHECKPOINT


,Qualifying_File,Status,Rows_Audited,Drivers,Session_Laps,Analytically_Eligible_Laps,Stint_Eligible_Laps,Laps_Outside_Source_Stint,Missing_LapTime,Non_Clear_Status,PitStop_Laps,Sector_Anomalies
0,bahrain-grand-prix-qualifying-4drivers-Tracing...,PASS,76,4,19,45,45,0,31,0,0,0
1,belgian-grand-prix-qualifying-4drivers-Tracing...,PASS,88,4,22,52,52,0,33,0,3,0
2,dutch-grand-prix-qualifying-4drivers-TracingIn...,PASS,72,4,18,38,38,0,33,0,1,0
3,emilia-romagna-grand-prix-qualifying-4drivers-...,PASS,84,4,21,42,42,0,39,3,1,0
4,hungarian-grand-prix-qualifying-4drivers-Traci...,PASS,104,4,26,50,50,0,49,4,1,0
5,italian-grand-prix-qualifying-4drivers-Tracing...,PASS,84,4,21,41,41,0,40,2,1,0
6,singapore-grand-prix-qualifying-4drivers-Traci...,PASS,72,4,18,30,30,0,41,1,0,0
7,spanish-grand-prix-qualifying-4drivers-Tracing...,PASS,72,4,18,34,34,0,37,1,0,0



BATCH SUMMARY
-------------------------------------------------------------------
Files processed: 8
PASS:            8
FAIL:            0

MODULE 04 — BATCH STATUS: PASS ✅


In [ ]:
print(quali_validated_batch_data.keys())


dict_keys(['bahrain-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv', 'belgian-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv', 'dutch-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv', 'emilia-romagna-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv', 'hungarian-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv', 'italian-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv', 'singapore-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-02.csv', 'spanish-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03.csv'])


In [ ]:
import numpy as np
from scipy.stats import linregress


# ------------------------------------------------------------
# STEP 1 — Get validated Spanish data
# ------------------------------------------------------------

spanish_key = next(
    name for name in quali_validated_batch_data
    if "spanish-grand-prix" in name.lower()
)

quali_df = quali_validated_batch_data[spanish_key]["analysis_df"].copy(deep=True)


# ------------------------------------------------------------
# STEP 2 — Get fastest eligible lap from each source stint
# ------------------------------------------------------------

stint_source = quali_df[
    quali_df["Analytical_Eligible_Stint"] == True
].copy()

stint_metrics = (
    stint_source
    .groupby(["Driver", "Stint"], as_index=False)
    .agg(
        Fastest_Lap=("LapTime", "min"),
        Valid_Laps=("LapTime", "count")
    )
)


# ------------------------------------------------------------
# STEP 3 — Calculate whole-session metrics per driver
# ------------------------------------------------------------

session_results = []

for driver in quali_df["Driver"].dropna().unique():

    driver_stints = (
        stint_metrics[
            stint_metrics["Driver"] == driver
        ]
        .sort_values("Stint")
        .reset_index(drop=True)
    )

    performance_points = driver_stints["Fastest_Lap"].dropna()

    # Number of performance-extraction laps
    attempt_count = len(performance_points)

    # Overall best qualifying lap
    best_lap = (
        performance_points.min()
        if attempt_count >= 1
        else np.nan
    )

    # Sample SD
    if attempt_count >= 2:
        lap_sd = performance_points.std(ddof=1)
    else:
        lap_sd = np.nan

    # Slope + R² from the performance-extraction points
    if attempt_count >= 3:
        x = np.arange(1, attempt_count + 1)
        y = performance_points.to_numpy()

        regression = linregress(x, y)

        slope = regression.slope
        r_squared = regression.rvalue ** 2
    else:
        slope = np.nan
        r_squared = np.nan

    session_results.append({
        "Driver": driver,
        "Performance_Attempts": attempt_count,
        "Best_Lap": best_lap,
        "LapTime_SD": lap_sd,
        "LapTime_Slope": slope,
        "R_Squared": r_squared
    })


session_metrics = pd.DataFrame(session_results)


# ------------------------------------------------------------
# STEP 4 — Display results
# ------------------------------------------------------------

print("=== DRIVER × SOURCE STINT ===")

display(
    stint_metrics
    .sort_values(["Driver", "Stint"])
    .reset_index(drop=True)
)

print("\n=== DRIVER × WHOLE QUALIFYING SESSION ===")

display(
    session_metrics
    .sort_values("Driver")
    .reset_index(drop=True)
)

=== DRIVER × SOURCE STINT ===


,Driver,Stint,Fastest_Lap,Valid_Laps
0,HAM,1.0,73.033,2
1,HAM,2.0,72.143,2
2,HAM,3.0,72.666,2
3,HAM,4.0,71.792,2
4,HAM,5.0,71.816,2
5,HAM,6.0,71.701,1
6,NOR,1.0,72.386,2
7,NOR,2.0,71.872,2
8,NOR,3.0,71.796,2
9,NOR,4.0,71.383,2



=== DRIVER × WHOLE QUALIFYING SESSION ===


,Driver,Performance_Attempts,Best_Lap,LapTime_SD,LapTime_Slope,R_Squared
0,HAM,6,71.701,0.543473,-0.243286,0.701367
1,NOR,4,71.383,0.411681,-0.308500,0.935919
2,RUS,5,71.703,0.303710,-0.178000,0.858738
3,VER,3,71.653,0.371371,-0.316500,0.726326


In [ ]:
# ============================================================
# QUALIFYING — MODULE 05: METRICS
# Batch Checkpoint — All 8 Qualifying Sessions
# ============================================================

import numpy as np
from scipy.stats import linregress


quali_metrics_batch_data = {}
metrics_batch_checkpoint = []


# ------------------------------------------------------------
# Process every qualifying session
# ------------------------------------------------------------

for quali_name, quali_data in quali_validated_batch_data.items():

    quali_df = quali_data["analysis_df"].copy(deep=True)


    # --------------------------------------------------------
    # 1. Fastest eligible lap from each source stint
    # --------------------------------------------------------

    stint_source = quali_df[
        quali_df["Analytical_Eligible_Stint"] == True
    ].copy()

    stint_metrics = (
        stint_source
        .groupby(["Driver", "Stint"], as_index=False)
        .agg(
            Fastest_Lap=("LapTime", "min"),
            Valid_Laps=("LapTime", "count")
        )
        .sort_values(["Driver", "Stint"])
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # 2. Whole-session metrics per driver
    # --------------------------------------------------------

    session_results = []

    for driver in quali_df["Driver"].dropna().unique():

        driver_stints = (
            stint_metrics[
                stint_metrics["Driver"] == driver
            ]
            .sort_values("Stint")
            .reset_index(drop=True)
        )

        performance_points = (
            driver_stints["Fastest_Lap"]
            .dropna()
        )

        attempt_count = len(performance_points)


        # Overall best qualifying lap
        best_lap = (
            performance_points.min()
            if attempt_count >= 1
            else np.nan
        )


        # Sample SD
        if attempt_count >= 2:
            lap_sd = performance_points.std(ddof=1)
        else:
            lap_sd = np.nan


        # Slope + R²
        if attempt_count >= 3:

            x = np.arange(1, attempt_count + 1)
            y = performance_points.to_numpy()

            regression = linregress(x, y)

            slope = regression.slope
            r_squared = regression.rvalue ** 2

        else:
            slope = np.nan
            r_squared = np.nan


        session_results.append({
            "Driver": driver,
            "Performance_Attempts": attempt_count,
            "Best_Lap": best_lap,
            "LapTime_SD": lap_sd,
            "LapTime_Slope": slope,
            "R_Squared": r_squared
        })


    session_metrics = (
        pd.DataFrame(session_results)
        .sort_values("Driver")
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # 3. Store M05 outputs
    # --------------------------------------------------------

    quali_metrics_batch_data[quali_name] = {
        "stint_metrics": stint_metrics,
        "session_metrics": session_metrics
    }


    # --------------------------------------------------------
    # 4. Batch checkpoint
    # --------------------------------------------------------

    metrics_batch_checkpoint.append({
        "File": quali_name,
        "Drivers": len(session_metrics),
        "Performance_Points": len(stint_metrics),
        "PASS": True
    })


# ------------------------------------------------------------
# Display checkpoint
# ------------------------------------------------------------

metrics_batch_checkpoint = pd.DataFrame(metrics_batch_checkpoint)

display(metrics_batch_checkpoint)

print("\n=== M05 BATCH CHECKPOINT ===")
print(f"Sessions processed: {len(metrics_batch_checkpoint)}")
print(f"PASS: {metrics_batch_checkpoint['PASS'].sum()}")
print(f"FAIL: {(~metrics_batch_checkpoint['PASS']).sum()}")

,File,Drivers,Performance_Points,PASS
0,bahrain-grand-prix-qualifying-4drivers-Tracing...,4,24,True
1,belgian-grand-prix-qualifying-4drivers-Tracing...,4,21,True
2,dutch-grand-prix-qualifying-4drivers-TracingIn...,4,21,True
3,emilia-romagna-grand-prix-qualifying-4drivers-...,4,23,True
4,hungarian-grand-prix-qualifying-4drivers-Traci...,4,25,True
5,italian-grand-prix-qualifying-4drivers-Tracing...,4,23,True
6,singapore-grand-prix-qualifying-4drivers-Traci...,4,20,True
7,spanish-grand-prix-qualifying-4drivers-Tracing...,4,18,True



=== M05 BATCH CHECKPOINT ===
Sessions processed: 8
PASS: 8
FAIL: 0


In [ ]:
# ============================================================
# QUALIFYING — MODULE 06: COMPARISON
# Reference Test: Spanish GP Qualifying
# ============================================================


# ------------------------------------------------------------
# STEP 1 — Get M05 outputs
# ------------------------------------------------------------

spanish_key = "spanish-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03 (1).csv"

quali_df = quali_validated_batch_data[spanish_key]["analysis_df"].copy(deep=True)

stint_metrics = quali_metrics_batch_data[spanish_key]["stint_metrics"].copy(deep=True)

session_metrics = quali_metrics_batch_data[spanish_key]["session_metrics"].copy(deep=True)


# ------------------------------------------------------------
# STEP 2 — Find each driver's final source stint
#         that contains a recorded LapTime
# ------------------------------------------------------------

recorded_laps = quali_df[
    quali_df["LapTime"].notna() &
    quali_df["Stint"].notna()
].copy()

final_stints = (
    recorded_laps
    .groupby("Driver")["Stint"]
    .max()
    .reset_index(name="Final_Source_Stint")
)


# ------------------------------------------------------------
# STEP 3 — Get fastest recorded lap in each final stint
#         for deriving qualifying result
# ------------------------------------------------------------

final_attempts = recorded_laps.merge(
    final_stints,
    on="Driver",
    how="inner"
)

final_attempts = final_attempts[
    final_attempts["Stint"] == final_attempts["Final_Source_Stint"]
].copy()

final_attempts = (
    final_attempts
    .groupby(
        ["Driver", "Final_Source_Stint"],
        as_index=False
    )
    .agg(
        Final_Stint_Best_Lap=("LapTime", "min")
    )
)


# ------------------------------------------------------------
# STEP 4 — Derive qualifying position
#         Fastest final-attempt lap = P1
# ------------------------------------------------------------

final_attempts = final_attempts.sort_values(
    "Final_Stint_Best_Lap"
).reset_index(drop=True)

final_attempts["Qualifying_Position"] = (
    final_attempts.index + 1
)


# ------------------------------------------------------------
# STEP 5 — Combine result with M05 session metrics
# ------------------------------------------------------------

comparison_metrics = session_metrics.merge(
    final_attempts[
        [
            "Driver",
            "Final_Source_Stint",
            "Final_Stint_Best_Lap",
            "Qualifying_Position"
        ]
    ],
    on="Driver",
    how="left"
)


# ------------------------------------------------------------
# STEP 6 — Display final comparison table
# ------------------------------------------------------------

comparison_metrics = comparison_metrics[
    [
        "Driver",
        "Qualifying_Position",
        "Performance_Attempts",
        "Best_Lap",
        "LapTime_SD",
        "LapTime_Slope",
        "R_Squared",
        "Final_Source_Stint",
        "Final_Stint_Best_Lap"
    ]
].sort_values(
    "Qualifying_Position"
).reset_index(drop=True)


display(comparison_metrics)

KeyError: 'spanish-grand-prix-qualifying-4drivers-TracingInsights.com-datatable-2026-09-03 (1).csv'

In [ ]:
# ============================================================
# QUALIFYING — MODULE 06: COMPARISON
# Batch Checkpoint — All 8 Qualifying Sessions
# ============================================================

quali_comparison_batch_data = {}
comparison_batch_checkpoint = []


# ------------------------------------------------------------
# Process every qualifying session
# ------------------------------------------------------------

for quali_name, quali_data in quali_validated_batch_data.items():

    quali_df = quali_data["analysis_df"].copy(deep=True)

    stint_metrics = (
        quali_metrics_batch_data[quali_name]["stint_metrics"]
        .copy(deep=True)
    )

    session_metrics = (
        quali_metrics_batch_data[quali_name]["session_metrics"]
        .copy(deep=True)
    )


    # --------------------------------------------------------
    # 1. Find each driver's final source stint containing
    #    a recorded LapTime
    # --------------------------------------------------------

    recorded_laps = quali_df[
        quali_df["LapTime"].notna() &
        quali_df["Stint"].notna()
    ].copy()

    final_stints = (
        recorded_laps
        .groupby("Driver")["Stint"]
        .max()
        .reset_index(name="Final_Source_Stint")
    )


    # --------------------------------------------------------
    # 2. Get fastest recorded lap in each final source stint
    # --------------------------------------------------------

    final_attempts = recorded_laps.merge(
        final_stints,
        on="Driver",
        how="inner"
    )

    final_attempts = final_attempts[
        final_attempts["Stint"] == final_attempts["Final_Source_Stint"]
    ].copy()

    final_attempts = (
        final_attempts
        .groupby(
            ["Driver", "Final_Source_Stint"],
            as_index=False
        )
        .agg(
            Final_Stint_Best_Lap=("LapTime", "min")
        )
    )


    # --------------------------------------------------------
    # 3. Derive qualifying position
    # --------------------------------------------------------

    final_attempts = (
        final_attempts
        .sort_values("Final_Stint_Best_Lap")
        .reset_index(drop=True)
    )

    final_attempts["Qualifying_Position"] = (
        final_attempts.index + 1
    )


    # --------------------------------------------------------
    # 4. Combine with M05 session metrics
    # --------------------------------------------------------

    comparison_metrics = session_metrics.merge(
        final_attempts[
            [
                "Driver",
                "Final_Source_Stint",
                "Final_Stint_Best_Lap",
                "Qualifying_Position"
            ]
        ],
        on="Driver",
        how="left"
    )


    comparison_metrics = comparison_metrics[
        [
            "Driver",
            "Qualifying_Position",
            "Performance_Attempts",
            "Best_Lap",
            "LapTime_SD",
            "LapTime_Slope",
            "R_Squared",
            "Final_Source_Stint",
            "Final_Stint_Best_Lap"
        ]
    ].sort_values(
        "Qualifying_Position"
    ).reset_index(drop=True)


    # --------------------------------------------------------
    # 5. Store M06 output
    # --------------------------------------------------------

    quali_comparison_batch_data[quali_name] = {
        "comparison_metrics": comparison_metrics,
        "stint_metrics": stint_metrics
    }


    # --------------------------------------------------------
    # 6. Checkpoint
    # --------------------------------------------------------

    comparison_batch_checkpoint.append({
        "File": quali_name,
        "Drivers": len(comparison_metrics),
        "Positions_Assigned": comparison_metrics[
            "Qualifying_Position"
        ].notna().sum(),
        "PASS": (
            len(comparison_metrics) == len(session_metrics)
            and
            comparison_metrics["Qualifying_Position"].notna().all()
        )
    })


# ------------------------------------------------------------
# Display checkpoint
# ------------------------------------------------------------

comparison_batch_checkpoint = pd.DataFrame(
    comparison_batch_checkpoint
)

display(comparison_batch_checkpoint)

print("\n=== M06 BATCH CHECKPOINT ===")
print(f"Sessions processed: {len(comparison_batch_checkpoint)}")
print(f"PASS: {comparison_batch_checkpoint['PASS'].sum()}")
print(f"FAIL: {(~comparison_batch_checkpoint['PASS']).sum()}")

,File,Drivers,Positions_Assigned,PASS
0,bahrain-grand-prix-qualifying-4drivers-Tracing...,4,4,True
1,belgian-grand-prix-qualifying-4drivers-Tracing...,4,4,True
2,dutch-grand-prix-qualifying-4drivers-TracingIn...,4,4,True
3,emilia-romagna-grand-prix-qualifying-4drivers-...,4,4,True
4,hungarian-grand-prix-qualifying-4drivers-Traci...,4,4,True
5,italian-grand-prix-qualifying-4drivers-Tracing...,4,4,True
6,singapore-grand-prix-qualifying-4drivers-Traci...,4,4,True
7,spanish-grand-prix-qualifying-4drivers-Tracing...,4,4,True



=== M06 BATCH CHECKPOINT ===
Sessions processed: 8
PASS: 8
FAIL: 0


In [ ]:
# ============================================================
# QUALIFYING — MODULE 07: EXPORT
# Final report-ready workbook
# ============================================================

import pandas as pd
from google.colab import files


# ------------------------------------------------------------
# Columns required for final export
# ------------------------------------------------------------

export_columns = [
    "Driver",
    "Qualifying_Position",
    "Performance_Attempts",
    "Best_Lap",
    "LapTime_SD",
    "LapTime_Slope",
    "R_Squared"
]


# ------------------------------------------------------------
# Create workbook
# ------------------------------------------------------------

output_filename = "F1_Qualifying_Analysis.xlsx"

with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:

    for quali_name, quali_data in quali_comparison_batch_data.items():

        export_df = quali_data["comparison_metrics"][
            export_columns
        ].copy()

        sheet_name = quali_name.split("-grand-prix")[0].title()

        export_df.to_excel(
            writer,
            sheet_name=sheet_name[:31],
            index=False
        )


print(f"Export created: {output_filename}")
print(f"Sessions exported: {len(quali_comparison_batch_data)}")


# ------------------------------------------------------------
# Download workbook
# ------------------------------------------------------------

files.download(output_filename)

Export created: F1_Qualifying_Analysis.xlsx
Sessions exported: 8


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>